In [ ]:
from pathlib import Path

import polars as pl

AIX_DIR = Path("../data/0_aix_downloads/high_compression")
FILTERED_DIR = Path("../data/1_filtered_games")
TOKENIZED_DIR = Path("../data/2_tokenized")

## Stage 0: Raw Aix (downloaded from HuggingFace)


In [2]:
# List available Aix files
aix_files = sorted(AIX_DIR.glob("*.parquet"))[:3]
print("Available Aix files:", len(list(AIX_DIR.glob("*.parquet"))))
print("Sample files:", [f.name for f in aix_files])

Available Aix files: 68
Sample files: ['aix_lichess_2013-01_high.parquet', 'aix_lichess_2013-02_high.parquet', 'aix_lichess_2013-03_high.parquet']


In [3]:
# Load one Aix file and inspect schema
aix_sample = pl.read_parquet(aix_files[0], n_rows=5)
print("Columns:", aix_sample.columns)
print("\nSample row:")
aix_sample

Columns: ['lichess_id', 'tournament', 'movedata', 'clocks_white', 'clocks_black', 'evals', 'ply_count', 'white', 'black', 'white_rating', 'black_rating', 'time_initial', 'time_increment', 'result', 'termination', 'white_rating_diff', 'black_rating_diff', 'eco', 'opening', 'white_title', 'black_title', 'utc_timestamp']

Sample row:


lichess_id,tournament,movedata,clocks_white,clocks_black,evals,ply_count,white,black,white_rating,black_rating,time_initial,time_increment,result,termination,white_rating_diff,black_rating_diff,eco,opening,white_title,black_title,utc_timestamp
str,str,binary,list[u16],list[u16],list[i16],u16,str,str,i16,i16,u16,u8,str,str,i16,i16,str,str,str,str,datetime[μs]
"""j1dkb5dw""",null,"b""\xc7\x02T\xd5\C`\x10\x07\xa3\x85#\xc4*8\x8a""",null,null,null,25,"""BFG9k""","""mamalak""",1639,1403,600,8,"""1-0""","""Normal""",5,-8,"""C00""","""French Defense: Normal Variati…",null,null,2012-12-31 23:01:03
"""a9tcp02g""",null,"b""\xed{\xc2\x88o\x1c\xc8^\xcao\x1aus\xb8\xd7/\x82""",null,null,null,35,"""Desmond_Wilson""","""savinka59""",1654,1919,480,2,"""1-0""","""Normal""",19,-22,"""D04""","""Queen's Pawn Game: Colle Syste…",null,null,2012-12-31 23:04:12
"""szom2tog""",null,"b""\x7fc\x0f\xacG0\x95\x16\xc0S\x0e\xac""",null,null,null,21,"""Kozakmamay007""","""VanillaShamanilla""",1643,1747,420,17,"""1-0""","""Normal""",13,-94,"""C50""","""Four Knights Game: Italian Var…",null,null,2012-12-31 23:03:15
"""rklpc7mk""",null,"b""\xb3\xce\x9b\x8fl\x0bK\xc9E8\x20\xf9\xca\x1fO\xc2c\xc2\xc4K`'I\xce_\xfd\x04\xe1I\xab\xd4xD\xfc\xe6\\xd1a\x8b7\x92r\xb6*\x94|>\xd9\x03\xbb""",null,null,null,94,"""Naitero_Nagasaki""","""800""",1824,1973,60,1,"""0-1""","""Normal""",-6,8,"""B12""","""Caro-Kann Defense: Goldman Var…",null,null,2012-12-31 23:04:57
"""1xb3os63""",null,"b""\x07\x18B\x89k\xaeg&\xe9,\x8c\x1b\x83sNSIK\x9fx\x8a}\x15\xa5\xdc/\xb2""",null,null,null,46,"""nichiren1967""","""Naitero_Nagasaki""",1765,1815,60,1,"""0-1""","""Normal""",-9,9,"""C00""","""French Defense: La Bourdonnais…",null,null,2012-12-31 23:02:37


## Stage 1: Filtered Games (UCI + evals + FEN)


In [4]:
# Load filtered games
filtered_files = sorted(FILTERED_DIR.glob("*.parquet"))[:3]
print("Filtered game files:", len(list(FILTERED_DIR.glob("*.parquet"))))
print("Sample files:", [f.name for f in filtered_files])

Filtered game files: 52
Sample files: ['eval_games_2013-01.parquet', 'eval_games_2013-02.parquet', 'eval_games_2013-03.parquet']


In [5]:
# Load sample filtered game
filtered_sample = pl.read_parquet(filtered_files[0], n_rows=5)
print("Columns:", filtered_sample.columns)
print("\nSample row:")
filtered_sample

Columns: ['lichess_id', 'uci_moves', 'evals_cp', 'evals_raw', 'is_check', 'is_capture', 'piece_moved', 'promotion', 'is_en_passant', 'white_rating', 'black_rating', 'result', 'game_end_reason', 'time_initial', 'time_increment', 'utc_timestamp', 'opening', 'eco', 'ply_count', 'fen']

Sample row:


lichess_id,uci_moves,evals_cp,evals_raw,is_check,is_capture,piece_moved,promotion,is_en_passant,white_rating,black_rating,result,game_end_reason,time_initial,time_increment,utc_timestamp,opening,eco,ply_count,fen
str,str,list[i16],list[i16],list[bool],list[bool],list[str],list[str],list[bool],i16,i16,str,str,u16,u8,datetime[μs],str,str,u16,str
"""2irq4pg0""","""e2e4 e7e5 g1f3 d7d6 d2d4 e5d4 …","[12, 26, … null]","[12, 26, … 32767]","[false, false, … true]","[false, false, … true]","[""p"", ""p"", … ""q""]","["""", """", … """"]","[false, false, … false]",1785,1944,"""1-0""","""mate""",300,0,2013-01-01 06:15:59,"""Philidor Defense: Exchange Var…","""C41""",43,"""rnbqkbnr/pppppppp/8/8/8/8/PPPP…"
"""cbgpp8cc""","""e2e4 e7e5 f1c4 b8c6 c2c3 g7g6 …","[22, 20, … null]","[22, 20, … -32768]","[false, false, … true]","[false, false, … false]","[""p"", ""p"", … ""p""]","["""", """", … """"]","[false, false, … false]",1538,1607,"""0-1""","""mate""",600,10,2013-01-01 20:08:24,"""Bishop's Opening""","""C23""",52,"""rnbqkbnr/pppppppp/8/8/8/8/PPPP…"
"""8rpcvpav""","""e2e4 d7d5 e4d5 d8d5 b1c3 d5a5 …","[22, 41, … 0]","[22, 41, … 0]","[false, false, … true]","[false, false, … false]","[""p"", ""p"", … ""q""]","["""", """", … """"]","[false, false, … false]",1565,1598,"""1/2-1/2""","""agreement""",300,0,2013-01-02 02:45:21,"""Scandinavian Defense: Main Lin…","""B01""",89,"""rnbqkbnr/pppppppp/8/8/8/8/PPPP…"
"""282s2n5n""","""e2e4 e7e5 c2c4 g8f6 d2d3 c7c6 …","[17, 30, … null]","[17, 30, … -32763]","[false, false, … false]","[false, false, … false]","[""p"", ""p"", … ""p""]","["""", """", … """"]","[false, false, … false]",1835,1812,"""0-1""","""resignation""",540,0,2013-01-02 02:58:57,"""English Opening: The Whale""","""C20""",106,"""rnbqkbnr/pppppppp/8/8/8/8/PPPP…"
"""etvda2dw""","""e2e4 e7e5 g1f3 b8c6 f1c4 g8f6 …","[27, 37, … -1533]","[27, 37, … -1533]","[false, false, … false]","[false, false, … true]","[""p"", ""p"", … ""q""]","["""", """", … """"]","[false, false, … false]",1692,1866,"""0-1""","""resignation""",300,0,2013-01-02 11:48:16,"""Italian Game: Two Knights Defe…","""C57""",28,"""rnbqkbnr/pppppppp/8/8/8/8/PPPP…"


## Stage 2: Tokenized (train/val ready)


In [6]:
# Load tokenized datasets
pretrain_path = TOKENIZED_DIR / "pretrain.parquet"
eval_path = TOKENIZED_DIR / "eval.parquet"

pretrain_sample = pl.read_parquet(pretrain_path, n_rows=5)
eval_sample = pl.read_parquet(eval_path, n_rows=5)

print("Pretrain shape:", pl.read_parquet(pretrain_path).shape)
print("Eval shape:", pl.read_parquet(eval_path).shape)
print("\nPretrain columns:", pretrain_sample.columns)
print("\nSample tokenized row:")
pretrain_sample

Pretrain shape: (5208849, 1)
Eval shape: (5296, 1)

Pretrain columns: ['token_ids']

Sample tokenized row:


token_ids
list[u16]
"[0, 3, … 1]"
"[0, 4, … 1]"
"[0, 3, … 1]"
"[0, 4, … 1]"
"[0, 4, … 1]"


## Tokenizer


In [7]:
from krasnal.tokenizer import Tokenizer

KRASNAL_DIR = Path("../src/krasnal")
uci_moves_path = KRASNAL_DIR / "uci_moves.txt"
tk = Tokenizer(uci_moves_path)
id_to_move = tk.id_to_move

### Opening analysis


In [8]:
# Load filtered games and analyze openings
raw_lf = pl.scan_parquet(str(FILTERED_DIR / "*.parquet"))

openings = (
    raw_lf.select("opening")
    .collect()["opening"]
    .str.split(":")
    .list.get(0)
    .str.split(",")
    .list.get(0)
    .str.replace(r"#\d+", "")
    .str.strip_chars()
    .str.replace_all(r"\s+", " ")
    .unique()
    .sort()
)

print("unique normalized openings:", len(openings))
for opening in openings:
    print(opening)

unique normalized openings: 170
Alekhine Defense
Amar Opening
Amazon Attack
Amsterdam Attack
Anderssen Opening
Australian Defense
Barnes Defense
Barnes Opening
Benko Gambit
Benko Gambit Accepted
Benko Gambit Declined
Benoni Defense
Bird Opening
Bishop's Opening
Blackmar-Diemer
Blackmar-Diemer Gambit
Blackmar-Diemer Gambit Declined
Blumenfeld Countergambit
Blumenfeld Countergambit Accepted
Boden-Kieseritzky Gambit
Bogo-Indian Defense
Borg Defense
Borg Opening
Bronstein Gambit
Budapest Defense
Canard Opening
Caro-Kann Defense
Carr Defense
Catalan Opening
Center Game
Center Game Accepted
Clemenz Opening
Colle System
Crab Opening
Creepy Crawly Formation
Czech Defense
Danish Gambit
Danish Gambit Accepted
Danish Gambit Declined
Doery Defense
Duras Gambit
Dutch Defense
East Indian Defense
Elephant Gambit
English Defense
English Opening
English Orangutan
English Rat
Englund Gambit
Englund Gambit Complex
Englund Gambit Complex Declined
Englund Gambit Declined
Formation
Four Knights
Four Knights

### Sequence statistics


In [9]:
# length of the longest game in raw data (by move count)
raw_with_lengths = raw_lf.select(
    pl.col("uci_moves").str.split(" ").list.len().alias("move_count"),
    pl.col("uci_moves"),
).collect()

max_length = raw_with_lengths["move_count"].max()
max_idx = raw_with_lengths["move_count"].arg_max()
longest_game_moves = raw_with_lengths[max_idx, "uci_moves"]

print("longest game length (moves):", max_length)
print("example longest game:")
print(longest_game_moves)

longest game length (moves): 597
example longest game:
d2d4 d7d5 c2c4 c7c6 b1c3 c8f5 g1f3 g8f6 e2e3 h7h6 f1e2 e7e6 e1g1 b8d7 b2b3 f8e7 c1b2 f6e4 a1c1 d7f6 c4d5 c6d5 c3e4 f5e4 f3e5 e8g8 a2a3 f6d7 b3b4 d7e5 d4e5 b7b6 b2d4 a8c8 d1a4 c8c1 f1c1 d8b8 a4d7 f8e8 e2b5 a7a5 d7e8 b8e8 b5e8 a5b4 a3b4 e7b4 d4b6 g8f8 e8b5 f7f5 e5f6 g7f6 b6c5 b4c5 c1c5 h6h5 c5c7 e4g6 h2h4 g6f7 f2f4 f8g7 b5e8 e6e5 e8f7 d5d4 f7h5 g7h8 e3d4 e5e4 g1f1 e4e3 f1e2 f6f5 e2e3 h8g8 e3d3 g8h8 d3c4 h8g8 c4b5 g8h8 b5b6 h8g8 b6b7 g8h8 b7a8 h8g8 a8a7 g8h8 a7a6 h8g8 a6a5 g8h8 a5a4 h8g8 a4a3 g8h8 a3a2 h8g8 a2a1 g8h8 a1b1 h8g8 b1c1 g8h8 c1d1 h8g8 d1e1 g8h8 e1f1 h8g8 f1g1 g8h8 g1h2 h8g8 h2g3 g8h8 g3f3 h8g8 f3e3 g8h8 e3d3 h8g8 d3c3 g8h8 c3b3 h8g8 b3b4 g8h8 b4b5 h8g8 h5g6 g8h8 g6f5 h8g8 b5b6 g8h8 b6b7 h8g8 b7a7 g8h8 a7a6 h8g8 a6a5 g8h8 a5a4 h8g8 a4a3 g8h8 a3a2 h8g8 a2a1 g8h8 a1b1 h8g8 b1c1 g8h8 c1d1 h8g8 d1e1 g8h8 e1f1 h8g8 f1g1 g8h8 g1h1 h8g8 h1h2 g8h8 h2h3 h8g8 h3g3 g8h8 g3f2 h8g8 f2e2 g8h8 e2d2 h8g8 d2c2 g8h8 c2b2 h8g8 b2b3 g8h8 b3b4 

In [11]:
# count number of <GAME> and </GAME> tokens in pretrain parquet
GAME_START_TOKEN_ID = tk.game_start_id
GAME_END_TOKEN_ID = tk.game_end_id

flat_tokens = pl.col("token_ids").explode()
counts = pretrain_sample.select(
    flat_tokens.eq(GAME_START_TOKEN_ID).sum().alias("game_start_count"),
    flat_tokens.eq(GAME_END_TOKEN_ID).sum().alias("game_end_count"),
)

print("Tokenized column checked: token_ids")
print("<GAME> token ID:", GAME_START_TOKEN_ID)
print("</GAME> token ID:", GAME_END_TOKEN_ID)
print("total <GAME> tokens in sample:", int(counts["game_start_count"][0]))
print("total </GAME> tokens in sample:", int(counts["game_end_count"][0]))

Tokenized column checked: token_ids
<GAME> token ID: 0
</GAME> token ID: 1
total <GAME> tokens in sample: 5
total </GAME> tokens in sample: 5


### Tokenized game example


In [12]:
# Example of a tokenized game with token IDs
example_game = pretrain_sample.head(1).to_dicts()[0]
token_ids = example_game["token_ids"]

print("Token IDs:", token_ids)
print("Total tokens:", len(token_ids))

Token IDs: [0, 3, 11, 11, 1146, 3547, 770, 1791, 3474, 2223, 734, 15, 525, 1112, 15, 1413, 1536, 1317, 1870, 3051, 1502, 1817, 2980, 1029, 1164, 931, 3394, 3251, 884, 1061, 3080, 321, 3300, 1451, 15, 788, 2197, 54, 1747, 3206, 1409, 1162, 1699, 1192, 981, 1526, 1087, 3002, 2863, 568, 1469, 458, 2139, 510, 1165, 3056, 433, 3186, 75, 1610, 2565, 15, 3014, 893, 1828, 763, 1896, 289, 2182, 3037, 56, 653, 470, 965, 15, 1210, 3227, 392, 2471, 30, 2115, 458, 3077, 530, 2847, 15, 224, 3227, 374, 563, 778, 2565, 15, 458, 1]
Total tokens: 95


In [13]:
# Same game decoded as string tokens
tokens_decoded = [id_to_move[tid] for tid in token_ids]

print("Tokens as strings:")
for i, token in enumerate(tokens_decoded):
    print(f"  {i}: {token}")

print(f"\nTotal tokens: {len(tokens_decoded)}")

Tokens as strings:
  0: <GAME>
  1: <win_white>
  2: <ELO_2000_2499>
  3: <ELO_2000_2499>
  4: wd2d4
  5: bg8f6
  6: wc2c4
  7: be7e6
  8: wg1f3
  9: bf8b4
  10: wc1d2
  11: <CHECK>
  12: bb4d2
  13: wd1d2
  14: <CHECK>
  15: bd7d5
  16: we2e3
  17: bd5c4
  18: wf1c4
  19: bb8d7
  20: we1g1
  21: be8g8
  22: wb1c3
  23: bc7c5
  24: wd2c2
  25: bc5d4
  26: wf3d4
  27: bd7b6
  28: wc4d3
  29: bc8d7
  30: wc3e4
  31: ba8c8
  32: we4f6
  33: bd8f6
  34: <CHECK>
  35: wc2d2
  36: bf8d8
  37: wa1c1
  38: be6e5
  39: wd4b3
  40: bd7c6
  41: wd2e2
  42: be5e4
  43: wd3b5
  44: bc6b5
  45: we2b5
  46: bc8c1
  47: wb3c1
  48: bh7h6
  49: wb5b3
  50: bd8d2
  51: wb3b4
  52: bf6b2
  53: wb4e4
  54: bd2c2
  55: wc1d3
  56: bb2a2
  57: wd3b4
  58: ba2c4
  59: we4e8
  60: bg8h7
  61: <CHECK>
  62: wb4c2
  63: bc4c2
  64: we8f7
  65: bc2c6
  66: wf1a1
  67: ba7a6
  68: wf7b3
  69: bb6d5
  70: wa1b1
  71: bb7b5
  72: wb3d3
  73: bc6g6
  74: <CHECK>
  75: wd3b3
  76: bd5c3
  77: wb1a1
  78: bg6f6
  79: 